In [4]:
from google.colab import drive
MAIN_FOLDER="/content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data"
DATA_FOLDER=MAIN_FOLDER+"/data"
CODE_FOLDER=MAIN_FOLDER+"/code"
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import sys
# Append the folder path to sys.path so Python can find the local modules
if MAIN_FOLDER not in sys.path:
    sys.path.append(MAIN_FOLDER)
    print(f'Added {MAIN_FOLDER} to sys.path')
if DATA_FOLDER not in sys.path:
    sys.path.append(DATA_FOLDER)
    print(f'Added {DATA_FOLDER} to sys.path')
if CODE_FOLDER not in sys.path:
    sys.path.append(CODE_FOLDER)
    print(f'Added {CODE_FOLDER} to sys.path')

Added /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data to sys.path
Added /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/data to sys.path
Added /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/code to sys.path


In [6]:
from __future__ import annotations

import torch
from visualize import extract_and_plot
import lm
import torch
from torch import nn, optim
from transformer import TransformerLM
import data
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 28.4 MB/s eta 0:00:00


In [28]:
print(torch.cuda.is_available())
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
seq_len = 128
batch_size = 64
data_path = DATA_FOLDER+"/he/"
n_layers = 6
n_heads = 6
embed_size = 192
mlp_hidden_size = embed_size * 4
learning_rate = 5e-4
gradient_clipping = 1.0
num_batches_to_train = 2000

True
Using device: cuda:0


In [8]:

tokenizer, tokenized_data = data.load_data(data_path)
# NOTE: are data items are longer by one than the sequence length,
# They will be shortened by 1 when converted to training examples.
data_iter = iter(data.RandomOrderDataIterator(tokenized_data, seq_len + 1))

In [9]:
model: torch.nn.Module = TransformerLM(
        n_layers,
        n_heads,
        embed_size,
        seq_len,
        tokenizer.vocab_size(),
        mlp_hidden_size,
        with_residuals=True,
    )

model = model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, betas=[0.9, 0.95])
num_batches = 0

Parameter count: 2.53M


In [25]:
import os

# Define the paths for the directories
models_dir = os.path.join(MAIN_FOLDER, 'models') # Use MAIN_FOLDER for relative path consistency
figs_dir = os.path.join(MAIN_FOLDER, 'figs')

# Check if the 'models' directory exists, and create it if not
if not os.path.exists(models_dir):
    os.makedirs(models_dir)
    print(f"Created directory: {models_dir}")
else:
    print(f"Directory already exists: {models_dir}")

# Check if the 'figs' directory exists, and create it if not
if not os.path.exists(figs_dir):
    os.makedirs(figs_dir)
    print(f"Created directory: {figs_dir}")
else:
    print(f"Directory already exists: {figs_dir}")

Directory already exists: /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models
Directory already exists: /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/figs


In [31]:

import optuna
import torch
import torch.optim as optim
from utils import save_best_model,parameters,loss_plotter,split_data
def objective(trial):
    # 1. Define the hyperparameter search space
    seq_len = trial.suggest_categorical("seq_len", [128, 256])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    n_layers = trial.suggest_categorical("n_layers", [4, 6, 8])
    n_heads = trial.suggest_categorical("n_heads", [4,8])
    embed_size = trial.suggest_categorical("embed_size", [128, 256, 384])

    # Assuming learning rate should be a log-uniform float. Adjust bounds as needed.
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    mlp_hidden_size = embed_size * 4

    params = {
        "seq_len": seq_len,
        "batch_size": batch_size,
        "n_layers": n_layers,
        "n_heads": n_heads,
        "embed_size": embed_size,
        "mlp_hidden_size": mlp_hidden_size,
        "learning_rate": learning_rate
    }

    # 2. Data Split and Iterators
    train_data, val_data = split_data(tokenized_data, seq_len)

    train_iter = iter(data.RandomOrderDataIterator(train_data, seq_len + 1))
    val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))

    print(f'Training model with params: {params}')

    # 3. Initialize Model
    model: torch.nn.Module = TransformerLM(
        params['n_layers'],
        params['n_heads'],
        params['embed_size'],
        params['seq_len'],
        tokenizer.vocab_size(),
        params['mlp_hidden_size'],
        with_residuals=True,
    )
    model = model.to(device)

    # 4. Setup Optimizer & Scheduler
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, betas=[0.9, 0.95])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5,
        threshold=1e-4,
        min_lr=1e-7
    )

    # 5. Training Loop Variables
    best_val_loss = float('inf')
    val_losses = []
    train_losses = []
    num_batches = 0
    early_stopping_patience = 5 # Number of validation checks to wait before stopping
    epochs_no_improve = 0
    model.train()

    for batch in data.batch_items(train_iter, batch_size):
        if num_batches >= num_batches_to_train:
            break

        batch_x, batch_y = lm.batch_to_labeled_samples(batch)
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        loss = lm.compute_loss(logits, batch_y)

        # Parameter update
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
        optimizer.step()

        num_batches += 1

        if num_batches % 10 == 0:
            # print(f"Seen {num_batches} batches. last loss is: {loss.item()}")
            train_losses.append(loss.item())

            # if num_batches % 100 == 0:
                # for _ in range(1):
                    # model.eval()
                    # sampled = tokenizer.detokenize(
                        # model.sample_continuation(tokenizer.tokenize("Hello"), 500)
                    # )
                    # model.train()
                    # print(f"Model sample: '''{sampled}'''")
                # print("")

        if num_batches % 100 == 0:
            model.eval()
            with torch.no_grad():
                val_batch = None
                try:
                    val_batch = next(data.batch_items(val_iter, batch_size))
                except StopIteration:
                    val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))
                    val_batch = next(data.batch_items(val_iter, batch_size))

                val_x, val_y = lm.batch_to_labeled_samples(val_batch)
                val_x = val_x.to(device)
                val_y = val_y.to(device)

                val_logits = model(val_x)
                val_loss = lm.compute_loss(val_logits, val_y)
                curr_val_loss = val_loss.item()
                val_losses.append(curr_val_loss)
                scheduler.step(curr_val_loss)

                if curr_val_loss < best_val_loss:
                    best_val_loss = curr_val_loss
                    epochs_no_improve = 0  # Reset patience counter
                    save_best_model(model, params, best_val_loss, epoch=num_batches, out_dir=models_dir)
                else:
                    epochs_no_improve += 1 # Increment patience counter
                # print(f"Val loss: {curr_val_loss} best val loss: {best_val_loss}")

            # --- OPTUNA PRUNING CHECK ---
            trial.report(curr_val_loss, num_batches)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
            # ---  TRADITIONAL EARLY STOPPING CHECK ---
            if epochs_no_improve >= early_stopping_patience:
                print("Early stopping triggered: Validation loss stagnated.")
                break # Breaks out of the training loop, ending this trial early
            model.train()
        if num_batches %500 ==0:
          print(f"batch: {num_batches} Val loss: {curr_val_loss} best val loss: {best_val_loss}")


    # Plot metrics at the end of the trial
    loss_plotter(train_losses, val_losses, params, figs_dir)

    # 6. Return the objective value for Optuna to minimize
    return best_val_loss

In [32]:
# Create study and optionally use a pruner (MedianPruner is standard)
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=500)
)

# Run for a specified number of trials
study.optimize(objective, n_trials=30)

# Output the best results
print("\nOptimization Finished!")
print("Best trial:")
best_trial = study.best_trial

print(f"  Value (Best Val Loss): {best_trial.value}")
print("  Best Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-04-15 13:10:01,753] A new study created in memory with name: no-name-b9f67796-0d74-498a-a03e-396ba3e88515


Training model with params: {'seq_len': 128, 'batch_size': 32, 'n_layers': 4, 'n_heads': 4, 'embed_size': 128, 'mlp_hidden_size': 512, 'learning_rate': 6.394710377132017e-05}
Parameter count: 0.78M
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq128_bs32_L4_H4_E128_M512_lr6p394710377em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq128_bs32_L4_H4_E128_M512_lr6p394710377em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq128_bs32_L4_H4_E128_M512_lr6p394710377em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq128_bs32_L4_H4_E128_M512_lr6p394710377em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/

[I 2026-04-15 13:10:47,885] Trial 0 finished with value: 2.2791483402252197 and parameters: {'seq_len': 128, 'batch_size': 32, 'n_layers': 4, 'n_heads': 4, 'embed_size': 128, 'learning_rate': 6.394710377132017e-05}. Best is trial 0 with value: 2.2791483402252197.


Saved plot to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/figs/loss_plot_seq128_bs32_L4_H4_E128_M512_lr6p394710377em05_2026-04-15_13-10-46.png
Training model with params: {'seq_len': 128, 'batch_size': 128, 'n_layers': 6, 'n_heads': 4, 'embed_size': 384, 'mlp_hidden_size': 1536, 'learning_rate': 6.758138294714874e-05}
Parameter count: 9.93M
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq128_bs128_L6_H4_E384_M1536_lr6p758138295em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq128_bs128_L6_H4_E384_M1536_lr6p758138295em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq128_bs128_L6_H4_E384_M1536_lr6p758138295em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse

[I 2026-04-15 13:23:01,939] Trial 1 finished with value: 1.970022439956665 and parameters: {'seq_len': 128, 'batch_size': 128, 'n_layers': 6, 'n_heads': 4, 'embed_size': 384, 'learning_rate': 6.758138294714874e-05}. Best is trial 1 with value: 1.970022439956665.


...

Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq128_bs32_L6_H4_E384_M1536_lr0p0002391592981.pth
Training model with params: {'seq_len': 256, 'batch_size': 64, 'n_layers': 8, 'n_heads': 4, 'embed_size': 256, 'mlp_hidden_size': 1024, 'learning_rate': 4.103377633804121e-05}
Parameter count: 5.94M
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq256_bs64_L8_H4_E256_M1024_lr4p103377634em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq256_bs64_L8_H4_E256_M1024_lr4p103377634em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq256_bs64_L8_H4_E256_M1024_lr4p103377634em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-

[I 2026-04-15 14:45:40,132] Trial 28 pruned. 


Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq256_bs64_L8_H4_E256_M1024_lr4p103377634em05.pth
Training model with params: {'seq_len': 128, 'batch_size': 128, 'n_layers': 6, 'n_heads': 4, 'embed_size': 384, 'mlp_hidden_size': 1536, 'learning_rate': 8.135487061373347e-05}
Parameter count: 9.93M
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq128_bs128_L6_H4_E384_M1536_lr8p135487061em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq128_bs128_L6_H4_E384_M1536_lr8p135487061em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq128_bs128_L6_H4_E384_M1536_lr8p135487061em05.pth
Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-

[I 2026-04-15 14:48:53,681] Trial 29 pruned. 


Saved best model to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models/best_model_seq128_bs128_L6_H4_E384_M1536_lr8p135487061em05.pth

Optimization Finished!
Best trial:
  Value (Best Val Loss): 1.844977617263794
  Best Params: 
    seq_len: 128
    batch_size: 32
    n_layers: 6
    n_heads: 4
    embed_size: 384
    learning_rate: 0.0007461754845310048


In [ ]:
model: torch.nn.Module = TransformerLM(
        n_layers,
        n_heads,
        embed_size,
        seq_len,
        tokenizer.vocab_size(),
        mlp_hidden_size,
        with_residuals=True,
    )

model = model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, betas=[0.9, 0.95])
num_batches = 0